# Personalizing Text-to-Image Diffusion Models

Authors:


*   Beatrice Vinciguerra 1933209
*   Gaia Rossi 1933502

## Project aim and selected papers

The aim of our project is to fine-tune a pre-trained diffusion model in such a way that the user can not only produce images giving a text caption as input, but also synthesize instantiations of their **own concept**, such as their items, their pets ecc.
To realize these fine-tuning we followed essetially two papers:


*   Multi-Concept Customization of Text-to-Image Diffusion, abbreviated as **Custom Diffusion** from now on
*   DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation, abbreviated as **DreamBooth** from now on

Both the papers focus on the idea of **fine-tuning** a pre-existing **diffusion model** for image generation (Stable Diffusion for Custom Diffusion and Imagen for DreamBooth) in such a way that, after the fine-tuning on a specific subject having access just to **few images** of it, it is able to produce images of that specific instance, distinguishing it from the more general class.

For example, if we fine-tuned the model on our specific motoribike (called for example "\<new1\> motorbike") then we want that when asking to produce a photo of it (ex. "A photo of a \<new1\> motorbike") it represents that specific motorbike.

These kind of conditioning is done in both the papers initializing the token corresponding to our concept with a **rare token**, in such a way that its representation is distinguished as a unique and specific one, and then learn the optimal one during the training.
This of course leads to the problem of **language drift**, that is the main challenge of this project, that essetially makes the diffusion model "forget" the more general class concepts. Thus, there's the risk that after fine-tuning the model and asking a generic "photo of a motorbike" all the results will look like the specific instance \<new1\> motorbike.

The two papers differ for two main points: the parameters training and the way they deal with the problem of language drift.
<br>
Indeed, Custom Diffusion updates just $W_{k}$ and $W_{v}$ parameters of the diffusion model, while DreamBooth updates all parameters.
<br>
For what concerns language drift, Custom Diffusion adopts **Data Augmentation** as regularization instrument, while DreamBooth uses the so called **Prior Preservation Loss**.
<br>
Another difference is that Custom diffusion experiments also with multi-concept customization, but we only focused on one concept at time.

For our project, due to some limitation better explained in the Limitations section, we chose to follow the same approach of Custom Diffusion for what concerns the diffusion model and the parameters to update, but decided to experiment and compare both approaches to language drift.

## Theoretical background and key concepts

###LDM
As said in the previous section, we start from a pre-trained model called Stable Diffusion, which is a **latent text-to-image diffusion model** (LDM) capable of generating photo-realistic images given any text input.

Generally, a LDM  (in the context of text-to-image generation) first compresses the image into the latent space, that we know generally being a lower-dimensional space in which our information are embedded mantaining just the key information of the image. The model then gradually adds noise to the image, and is trained to denoise it and regenerate the image from scratch.
An LDM consists of a VAE, a modified U-Net and a text encoder.

###VAE, U-Net and Text Encoder
**VAE** stands for Variational Autoencoder and is a kind of latent generative model based on two networks:

*   **recognition model** (encoder), whose role is to learn the latent representation z for the input images x
*   **generative model** (decoder), that generates an image starting from the latent space

The generative model is essentially a **Bayesian network** of the shape $p(x|z)p(z)$, while the recognition model is a Bayesian network as well, but with $q(z|x)$, that is an approximation of $p(z|x)$ since it can't be known. This approximation is computed through **variational inference**, from which the name of this model.

A **text encoder** is a model that, after tokenizing an input text, transforms the obtained tokens into some kind of embedding.
<br>
The text encoder used by Stable Diffusion is the CLIP text encoder, that in turn is based on the use of transformers. It is used in Stable Diffusion to transform text prompts into embeddings for image generation.

A **U-Net** is a CNN specifically developed for image segmentation and is generally employed in diffusion models for the iterative image denoising phase.

In the context of LDMs, the encoder of the VAE compresses the images into a lower-dimentional latent space. Then Gaussian noise is iteratively injected in the latent representation in multiple steps. The U-Net denoises the output of this block until getting back to a latent representation.
Finally, the VAE starting from the obtained latent representation generates through its decoder the final image. The text encoder is used to transform text prompts into embeddings, which in turn are used to condition the denoising step in the U-Net through a cross-attention mechanism.

###Cross-attention
In the context of our project, the cross-attention block modifies the latent features of the network according to the text features. Assuming $c$ to be the text embedding and $f$ to be the latent image representation, a single-head cross-attention operation consists of:

$Q = W_{q}f \quad K = W_{k}c \quad V = W_{v}c$

$Cross$-$Attention(Q,K,V)=Softmax(\frac{QK^{T}}{\sqrt{d}})V$

Since as we explained before our fine-tuning focuses on the way in which we map text to image distribution and since the text embedding is only input to $W_{k}$ and $W_{v}$ projection matrices, then as suggested in the Custom Diffusion paper we can update just the parameters of these two during the fine-tuning process.


###Language drift

As we said in the previous section, the fine-tuning of layers that are conditioned on the text embeddings leads to the problem of language drift. Indeed language drift has been an observed problem in language models, where a model that is pre-trained on a large dataset of texts and later fine-tuned for a specific task progressively loses syntactic and semantic knowledge of the language.
<br>
In a similar way, fine-tuned diffusion models can slowly forget how to generate subjects of the same class as the target subject. Another related problem is the possibility of reduced output diversity.

To mitigate these two issues, Custom Diffusion and DreamBooth papers respectively propose these solutions:
*   the use of **regularization images** and corresponding caption taken from a dataset (LAION-400M in their paper)
*   **prior preservation loss**, which is based on the use of images generated by the pre-trained model before the fine-tuning as regularization images


###Data augmentation
Since our project aims to be able to produce images of a specific subject having just a few images of it, data augmentation is essential to be sure that the pre-trained model is properly fine-tuned.
<br>
Indeed data augmentation is based on the idea of applying a set of transformations (zoom in, zoom out, flip ecc.) to the images in order to produce a larger collection of them.

This technique was used just in Custom Diffusion, but for sake of curiosity we decided to use it also with the prior preservation loss approach.


###Cosine similarity

For the evaluation we used two kinds of metrics: text alignment and image alignment. Both these metrics are based on the use of cosine similarity.
<br>
It measures the angle between two vectors, indicating how similar they are in direction and this is pretty useful when comparing embeddings in the latent space. A higher value means greater similarity.

It is computed as:

$cos(θ)=\frac{a\,·\, b}{||a||\,\cdotp \,||b||}$

## Implementation details: dataset, models, experimental setup

The project implements two modalities of execution:


* custom, in which the regularization images are taken from an external dataset, the COCO one;
* prior, in which the regularization images are generated by the diffusion model given in input the prompt "a photo of a *class_name*".





In [ ]:
#choose a mode: either custom (default) or prior
mode = "custom"

if mode not in ["custom", "prior"]:
    raise ValueError("Mode must be either 'custom' or 'prior'")

In [ ]:
# variables for motorbike
instance_directory = "/content/drive/MyDrive/custom_diffusion/benchmark_dataset/transport_motorbike1"
instance_caption =  "photo of a <new1> motorbike"
class_prompt= "a photo of a motorbike"
class_name = "motorbike"

if mode == "custom":
  regularization_directory = "/content/drive/MyDrive/custom_diffusion/regularization/transport_motorbike"
  regularization_captions_file = "/content/drive/MyDrive/custom_diffusion/regularization/transport_motorbike/captions.txt"
  weights_dir = "/content/drive/MyDrive/custom_diffusion/weights/custom/motorbike1_exp3_params.pth"
else:
  regularization_directory = "/content/drive/MyDrive/custom_diffusion/prior_preservation/regularization/transport_motorbike"
  weights_dir = "/content/drive/MyDrive/custom_diffusion/weights/prior/motorbike1_exp3_params.pth"

**Regularization images download**: Coco dataset

We initially planned to collect regularization images from the same source of Custom Diffusion: LAION-400M. These dataset is widely used in this field, since it provides over 400 milion images with respective captions. We tried accessing it through a remote get operation, collecting images by caption research, but we found out that the dataset is recently gone offline.
<br> Since it was not feasible to download it due to its large size, we opted for another famous dataset called COCO.<br>
In order to use it, the dataset needs to be downloaded in its entirety as done in the Reproducibility section.

The following code retrieves for each class contained in the `dataset.json` file images containing the class name in the caption. Then they are saved in the `regularization` folder, that already contains all the images needed for each class when using our `custom_diffusion` folder



In [ ]:
import os, json, random, requests
from pycocotools.coco import COCO
from PIL import Image
from io import BytesIO
import re

def retrieve_from_coco_api(class_name, out_dir, num_class_images=200,
                           ann_file="/content/drive/MyDrive/custom_diffusion/coco/annotations/captions_train2017.json",
                           img_dir="/content/drive/MyDrive/custom_diffusion/coco/train2017"):
    """
    Retrieve up to num_class_images from COCO captions dataset
    where captions mention class_name. Save to out_dir.
    """
    os.makedirs(out_dir, exist_ok=True)

    coco = COCO(ann_file)

    # load all captions
    anns = coco.loadAnns(coco.getAnnIds())
    pattern = re.compile(rf"\b{re.escape(class_name.lower())}\b")
    matches = [ann for ann in anns if pattern.search(ann["caption"].lower())]
    print(f"[INFO] Found {len(matches)} COCO caption matches for '{class_name}'")

    chosen = random.sample(matches, min(num_class_images, len(matches)))

    captions = []
    count = 0
    for ann in chosen:
        img_id = ann["image_id"]
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])

        try:
            img = Image.open(img_path).convert("RGB")
            save_path = os.path.join(out_dir, f"{count}.jpg")
            img.save(save_path, "JPEG")

            captions.append(ann["caption"])
            count += 1
        except Exception as e:
            print(f"[WARN] Could not process {img_path}: {e}")

        if count >= num_class_images:
            break

    with open(os.path.join(out_dir, "captions.txt"), "w") as f:
        for c in captions:
            f.write(c.strip() + "\n")

    print(f"[DONE] Saved {count} images for '{class_name}' into {out_dir}")

def regenerate_all_class_images_from_coco_api(dataset_json, num_class_images=200,
                                              ann_file="/content/drive/MyDrive/custom_diffusion/coco/annotations/captions_train2017.json",
                                              img_dir="/content/drive/MyDrive/custom_diffusion/coco/train2017"):
    with open(dataset_json, "r") as f:
        entries = json.load(f)

    for entry in entries:
        class_prompt = entry["class_prompt"]  # e.g. "a photo of a dog"
        class_name = re.sub(r"^a photo of an?\s+", "", class_prompt, flags=re.IGNORECASE).strip()

        denormalized_path = os.path.join("./", entry["class_data_dir"])
        out_dir = os.path.normpath(denormalized_path)

        retrieve_from_coco_api(class_name, out_dir, num_class_images=num_class_images,
                               ann_file=ann_file, img_dir=img_dir)



# Example usage:
regenerate_all_class_images_from_coco_api("/content/drive/MyDrive/custom_diffusion/customconcept101/dataset.json", num_class_images=200)

**Regularization images download**: Prior preservation loss

In the following code the regularization images are generated using the Stable Diffusion model with its original weights, so no conditioning in place.


In [ ]:
import torch
import os
from diffusers import StableDiffusionPipeline

def getRegImages(
    class_prompt,
    out_dir,
    num_images=200,
    batch_size=2,
    num_inference_steps=50,
    guidance_scale=7.5,
    model_id="CompVis/stable-diffusion-v1-4"
):
    """
    Generate regularization images for prior preservation loss (DreamBooth style).
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float32).to(device)

    os.makedirs(out_dir, exist_ok=True)

    counter = 0
    while counter < num_images:
        current_bs = min(batch_size, num_images - counter)
        images = pipe(
            [class_prompt] * current_bs,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale
        ).images

        for img in images:
            img_path = os.path.join(out_dir, f"{counter:04d}.png")
            img.save(img_path)
            counter += 1

    print(f"Saved {num_images} images to {out_dir}")

In [ ]:
# Generate 200 generic <class_name> images for prior preservation
getRegImages(
    class_prompt=class_prompt,
    out_dir= regularization_directory,
    num_images=200
)

**Concept dataset creation**

In this next step, the dataset to feed the network for training is created. It contains a mix of instance images (those of the object we want the model to learn) and regularization images. The proportion of the two collections depends on the variable `ratio`.<br>
Data augmentation is performed in this phase in the same way it is done in the Custom Diffusion paper:

> During training, we randomly resize the target images to 1.2-1.4x every 1 out of 3 times and append *zoomed_in* or *close_up* to the text prompt. The rest of the time the target image is randomly resized to 0.4-1.0x and if the resize is less than 0.6 we append *far_away* or *very_small* to the text prompt.

The difference between the custom and prior approach is once again in the regularization images treatement:


*   in the custom diffusion approach, the images come with their own descriptive caption and this is kept in the dataset,
*   in the prior preservation approach, the images are equipped with the same caption given in input to the Stable Diffusion model in order to generate them ("a photo of a *class_name*").





In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import os
import random
class ConceptDataset(Dataset):
    def __init__(self, instance_dir, reg_dir, instance_caption, reg_captions_file=None, size=512, flip_p=0.5, ratio=2, mode="prior"):
        self.mode = mode
        self.ratio = ratio
        # Regularization images
        self.reg_images = sorted([os.path.join(reg_dir, f) for f in os.listdir(reg_dir) if f.endswith(("png","jpg","jpeg"))]) if reg_dir else []
        print(self.reg_images)
        #len of regularization images
        self.reg_length = len(self.reg_images)

        #Instance images
        self.instance_images = [os.path.join(instance_dir, f) for f in os.listdir(instance_dir) if f.endswith(("png","jpg","jpeg", "JPG"))]
        # len of instance images
        if self.reg_length < len(self.instance_images):
          self.instance_length = len(self.instance_images)
        else:
          self.factor = self.reg_length // (self.ratio*len(self.instance_images))
          self.remainder = self.reg_length % (len(self.instance_images))
          self.instance_length = len(self.instance_images) * self.factor + self.remainder
        print("instance length: " + str(self.instance_length))

        # Instance caption: "A photo of a <new1> dog."
        self.instance_caption = instance_caption
        # Regularization captions
        if reg_captions_file and os.path.exists(reg_captions_file):
            with open(reg_captions_file, "r") as f:
                self.reg_captions = [line.strip() for line in f.readlines()]
        else:
            self.reg_captions = []

        self.size = size
        self.flip = transforms.RandomHorizontalFlip(p=flip_p)

    def __len__(self):
        return len(self.instance_images) + len(self.reg_images)

    def __getitem__(self, i):
        example = {}
        #print("item " + str(i))
        # Choose instance vs regularization
        use_instance = (i % (self.instance_length + self.reg_length)) < self.instance_length

        if use_instance:
          # Instance sample
          img_path = self.instance_images[i % len(self.instance_images)]
          image = Image.open(img_path)
          example["caption"] = self.instance_caption
        else:
          #Regularization sample
          img_path = self.reg_images[i % len(self.reg_images)]
          image = Image.open(img_path)
          if self.mode == "custom":
            example["caption"] = "A photo of " + self.reg_captions[i % min(self.reg_length, len(self.reg_captions))]
          else: #mode == "prior"
            example["caption"] = class_prompt

        if image.mode != "RGB":
            image = image.convert("RGB")

        # crop of the image
        img = np.array(image).astype(np.uint8)
        crop = min(img.shape[0], img.shape[1])
        h, w, = img.shape[0], img.shape[1]

        img = img[(h - crop) // 2:(h + crop) // 2,
                  (w - crop) // 2:(w + crop) // 2]
        # flip of the image
        image = Image.fromarray(img)
        image = self.flip(image)

        #resize of the image
        # Augment only instance images: zoom in/out logic
        if use_instance:
            if random.randint(0, 2) < 2:
                # mostly downscale or same size
                random_scale = random.randint(self.size // 3, self.size)
            else:
                # sometimes upscale (close-up)
                random_scale = random.randint(int(1.2 * self.size), int(1.4 * self.size))
            if random_scale % 2 == 1:
                random_scale += 1
        else: # regularization case
            random_scale = self.size

        if use_instance and random_scale < int(0.6 * self.size):
            # ZOOM OUT: place a smaller image onto a black canvas
            add_to_caption = random.choice(["a far away ", "very small "])
            example["caption"] = add_to_caption + example["caption"]

            cx = random.randint(random_scale // 2, self.size - random_scale // 2)
            cy = random.randint(random_scale // 2, self.size - random_scale // 2)

            small = image.resize((random_scale, random_scale), resample= Image.BICUBIC)
            small = (np.array(small).astype(np.float32) / 127.5 - 1.0)

            canvas = np.zeros((self.size, self.size, 3), dtype=np.float32)
            x0, x1 = cx - random_scale // 2, cx + random_scale // 2
            y0, y1 = cy - random_scale // 2, cy + random_scale // 2
            canvas[x0:x1, y0:y1, :] = small
            input_image = canvas

            mask = np.zeros((self.size // 8, self.size // 8), dtype=np.float32)
            # +1/-1 margins to be conservative
            mask[(x0 // 8) + 1:(x1 // 8) - 1, (y0 // 8) + 1:(y1 // 8) - 1] = 1.0

        elif use_instance and random_scale > self.size:
            # ZOOM IN: resize bigger then crop 512x512 window
            add_to_caption = random.choice(["zoomed in ", "close up "])
            example["caption"] = add_to_caption + example["caption"]

            big = image.resize((random_scale, random_scale), resample= Image.BICUBIC)
            big = (np.array(big).astype(np.float32) / 127.5 - 1.0)

            cx = random.randint(self.size // 2, random_scale - self.size // 2)
            cy = random.randint(self.size // 2, random_scale - self.size // 2)

            x0, x1 = cx - self.size // 2, cx + self.size // 2
            y0, y1 = cy - self.size // 2, cy + self.size // 2
            input_image = big[x0:x1, y0:y1, :]
            mask = np.ones((self.size // 8, self.size // 8), dtype=np.float32)

        else: # regularization case
            # No special zoom: just resize to self.size(512)
            image = image.resize((self.size, self.size), resample= Image.BICUBIC)
            input_image = (np.array(image).astype(np.float32) / 127.5 - 1.0) #[-1, 1]
            mask = np.ones((self.size // 8, self.size // 8), dtype=np.float32)

        return {"image": input_image, "mask": mask, "caption": example["caption"]}

In [ ]:
from torch.utils.data import Dataset, ConcatDataset, DataLoader
if mode == "custom":
  print(f"Processing entry:")
  print(f"  Instance caption: {instance_caption}")
  print(f"  Instance Directory: {instance_directory}")
  print(f"  Regularization captions: {regularization_captions_file}")
  print(f"  Regularization Directory: {regularization_directory}")

  try:
    concept_dataset = ConceptDataset(
        instance_dir = instance_directory,
        reg_dir = regularization_directory,
        instance_caption = instance_caption,
        reg_captions_file = regularization_captions_file,
        size = 512,  # Image size
        flip_p = 0.5, # Probability of horizontal flip
        mode = mode
    )
  except FileNotFoundError as e:
      print(f"Error: {e}. Please check if the provided directories and files exist.")
  except Exception as e:
      print(f"An error occurred: {e}")
else: # mode == "prior"
  print(f"Processing entry:")
  print(f"  Instance caption: {instance_caption}")
  print(f"  Instance Directory: {instance_directory}")
  print(f"  Class Prompt: {class_prompt}")
  print(f"  Regularization Directory: {regularization_directory}")

  try:
    concept_dataset = ConceptDataset(
        instance_dir=instance_directory,
        reg_dir=regularization_directory,
        instance_caption= instance_caption,
        size=512,  # Image size
        flip_p=0.5, # Probability of horizontal flip
        mode= mode
    )
  except FileNotFoundError as e:
      print(f"Error: {e}. Please check if the provided directories and files exist.")
  except Exception as e:
      print(f"An error occurred: {e}")

**Fine-tuning of the model**

This cells fine-tune the Stable Diffusion model to associate the new rare token (\<new1\>) with the specific concept (e.g., the \<new1\> motorbike) using the provided instance and regularization images.

Before actually training the diffusion model, the following steps need to be done:
1.   **initialization**: the pre-trained Stable Diffusion model components (VAE, UNet, tokenizer, text encoder) and the noise scheduler are loaded;
2.   **freezing parameters**: all the parameters of every component of Stable Diffusion are frozen;
3.   **adding new token embedding**: the token that is associated to \<new1\> before the fine-tune is added to the tokenizer (if not already present) and we make sure that the token-id 42170 (corresponding to a rare token, as shown in the Custom Diffusion paper) is associated to \<new1\>;
4.   **enabling gradients**: only gradients for cross attention matrices $W_{k}$ and $W_{v}$ and for the weights of the embedding of our rare token are allowed (and thus they will be updated);
5.   **optimizer setup**: the AdamW optimizer is set to update only the trainable parameters. We used a learning rate of $4*10^{-5}$ so that it matches the learning rate used in Custom Diffusion (they had a batch size of $8$ and a learning rate $10^{-5}$ resulting in an effective learning rate $8*10^{-5}$, while we were forced to use a batch size of $2$).
6.   **data loader**: a data loader to iterate through the ConceptDataset during training is created.

Now, everything is ready to enter the training loop for `max_step` iterations:
* it prepares a batch of images and captions.
* it encodes the captions into text embeddings using the text encoder.
* it encodes the images into latents using the VAE.
* it adds noise to the latents at a random timestep.
* it uses the UNet to predict the noise.
* it calculates the mean squared error (MSE) loss between the predicted noise and the actual noise.
* it performs backpropagation and updates the trainable parameters using the optimizer.

After training, it saves the updated text encoder embedding weights and the UNet's $W_{k}$ and $W_{v}$ weights are saved into the file `weights_dir`.





In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDPMScheduler
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm # Import tqdm
from itertools import cycle


device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"  # or runwayml/stable-diffusion-v1-5

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float32).to(device)
vae, unet, tokenizer, text_encoder = pipe.vae, pipe.unet, pipe.tokenizer, pipe.text_encoder
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Freeze everything first
for m in (vae, unet, text_encoder):
    for p in m.parameters():
        p.requires_grad = False


# Add one new token
rare_token = "<new1>"
init_id = 42170   # CLIP embedding row to copy from

# Save original embeddings before resize
orig_emb = text_encoder.get_input_embeddings().weight.detach().clone()

# If the token is unknown for the tokenizer, then add it
if tokenizer.convert_tokens_to_ids(rare_token) == tokenizer.unk_token_id:
    tokenizer.add_tokens([rare_token])
    text_encoder.resize_token_embeddings(len(tokenizer), mean_resizing=False)

# Initialize embedding row
emb = text_encoder.get_input_embeddings()
new_id = tokenizer.convert_tokens_to_ids(rare_token) # id assigned to rare token
with torch.no_grad():
    emb.weight[new_id].copy_(orig_emb[init_id]) # replace with desired id (42170)

# Use a mask to allow gradient (and thus update) only the row related to the rare token
emb.weight.requires_grad_(True)
grad_mask = torch.zeros_like(emb.weight)
grad_mask[new_id] = 1.0
emb.weight.register_hook(lambda g: g * grad_mask.to(g.device))

# Allow gradient (and thus update) just for cross attention matrices Wk and Wv
cross_params = []
for name, module in unet.named_modules():
    is_cross = ("attn2" in name.lower()) or getattr(module, "is_cross_attention", False)
    if is_cross and hasattr(module, "to_k") and hasattr(module, "to_v"):
        for p in module.to_k.parameters():
            p.requires_grad = True
            cross_params.append(p)
        for p in module.to_v.parameters():
            p.requires_grad = True
            cross_params.append(p)

print(f"[INFO] Trainable tensors in UNet (Wk/Wv): {len(cross_params)}")

# list of parameters to optimize during training
opt_params = list({id(p): p for p in (cross_params + [emb.weight])}.values())
# initialization of the AdamW optimizer(<list of parameters that the optimizer will update>, <learning rate>)
optimizer = torch.optim.AdamW(opt_params, lr=4e-5)  # LR per paper

# shuffle: the data will be shuffled at the beginning of each epoch
# num_workers: subprocesses that will be used to load data. (max number for this system = 2)
# drop_last: ensures that if the total number of samples in the last bacth is not batch_size,
#            the last batch will be dropped
train_loader = DataLoader(concept_dataset, batch_size=2, shuffle=True, num_workers=2, drop_last=True)

scaling = 0.18215  # SD v1 latent scaling

def batch_to_device(batch, device):
    # Ensure shape is [B,3,H,W] and dtype float32 in [-1,1]
    px = batch["image"]
    if isinstance(px, np.ndarray):
        px = torch.from_numpy(px)
    if px.ndim == 4 and px.shape[-1] == 3:     # [B,H,W,3] -> [B,3,H,W]
        px = px.permute(0,3,1,2)
    elif px.ndim == 3 and px.shape[-1] == 3:   # [H,W,3] -> [1,3,H,W]
        px = px.permute(2,0,1).unsqueeze(0)
    return px.to(device).float(), batch["caption"]


max_step = 500 # number of training steps
global_step = 0

# Wrap the train_loader with tqdm
progress_bar = tqdm(train_loader, total=max_step, desc="Training")


for raw in cycle(progress_bar): # Iterate directly over the wrapped train_loader
    if global_step >= max_step:
        break

    # ---- prepare batch ----
    pixel_values, captions = batch_to_device(raw, device)

    # ---- text conditioning ----
    tok = tokenizer(
        list(captions),
        padding="max_length",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    ).to(device)

    encoder_hidden_states = text_encoder(tok.input_ids)[0]   # [B,77,dim]

    # ---- encode images to latents ----
    latents = vae.encode(pixel_values).latent_dist.sample() * scaling  # [B,4,64,64]

    # ---- pick random timestep and add noise ----
    noise = torch.randn_like(latents)
    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    # ---- predict noise with UNet ----
    noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states).sample

    # ---- loss & update ----
    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    global_step += 1
    # Update tqdm description with the loss
    progress_bar.set_postfix({"loss": loss.item()})


print(f"Finished training. Total steps: {global_step}")

# Save checkpoint

save = {}
save["text_encoder_emb_weight"] = text_encoder.get_input_embeddings().weight.detach().cpu()

unet_state = unet.state_dict()
wk_wv_only = {k: v.cpu() for k, v in unet_state.items() if ("to_k." in k) or ("to_v." in k)}
save["unet_wk_wv"] = wk_wv_only

torch.save(save, weights_dir)
print(f"Saved to {weights_dir}")


**Loading of the model**

In order to generate images of the \<new1\> object, the model needs to be loaded again together with the new weights just fine-tuned.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDPMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"  # or runwayml/stable-diffusion-v1-5

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float32).to(device)
vae, unet, tokenizer, text_encoder = pipe.vae, pipe.unet, pipe.tokenizer, pipe.text_encoder
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

ckpt = torch.load(weights_dir, map_location="cpu")

# Add the new token to the tokenizer and resize the text encoder embeddings
# This needs to be done *before* loading the saved embedding weights
rare_token = "<new1>"
if rare_token not in tokenizer.get_vocab():
    tokenizer.add_tokens([rare_token])
    text_encoder.resize_token_embeddings(len(tokenizer), mean_resizing=False)


# restore embedding (it contains your V*)
with torch.no_grad():
    emb = text_encoder.get_input_embeddings()
    emb.weight.copy_(ckpt["text_encoder_emb_weight"].to(emb.weight.device))

# restore UNet Wk/Wv
u_sd = unet.state_dict()
for k, v in ckpt["unet_wk_wv"].items():
    if k in u_sd:
        u_sd[k].copy_(v.to(u_sd[k].device))
unet.load_state_dict(u_sd)

**Generation of images**

The following line of code

```
image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]
```

is generating an image using the fine-tuned Stable Diffusion model. In particular:
* `prompt` is the text description of the image you want to generate.
* `num_inference_steps=50` controls the number of steps the diffusion process takes to generate the image. More steps generally lead to higher quality images but take longer to generate.
* `guidance_scale=7.5` influences how strongly the generated image is guided by the text prompt. Higher values result in images that are more closely aligned with the prompt, while lower values allow for more creativity and diversity.
* `.images`: the pipe object returns an object that contains a list of generated images and `[0]` selects the first (and only) image from the list.

What changes in the following two cells is that the first one is being used to generate 100 images and save them into a folder in order to later evaluate how the model performed while the second one is just generating one image at time and displaying it in the output of the cell.


In [ ]:
if mode == "custom"
  gen_dir = "/content/drive/MyDrive/custom_diffusion/generated_images/motorbike1_exp3"
else: # mode == "prior"
  gen_dir = "/content/drive/MyDrive/custom_diffusion/prior_preservation/generated_images/motorbike1_exp3"
prompt_file = "/content/drive/MyDrive/custom_diffusion/customconcept101/prompts/transport.txt"

Generation of 100 images for evaluation purposes

In [ ]:
from itertools import cycle
import os

# Open the file and read lines into a list
with open(prompt_file, 'r') as f:
    prompts = f.readlines()

# Optional: Remove trailing newline characters from each line
prompts = [prompt.strip() for prompt in prompts]

# Ensure the generation directory exists
os.makedirs(gen_dir, exist_ok=True)

i = 0
for prompt in cycle(prompts):
  if i == 100:
    break
  # Add the class to the prompt instead of {}
  prompt = prompt.replace("{}", f"<new1> {class_name}")

  # Generate an image
  # You can adjust parameters like num_inference_steps and guidance_scale
  image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]

  # Display the generated image
  image.save(os.path.join(gen_dir, f"{class_name}_{i}.png"))
  #display(image)
  i = i + 1

Generation of one image for a user chosen prompt

In [ ]:
# Define your prompt using the new token
prompt = "A photo of a cup."

# Generate an image
# You can adjust parameters like num_inference_steps and guidance_scale
image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]

# Display the generated image
#image.save("test_custom_diffusion.png")
display(image)

**Text alignment**

Text alignment quantifies how well the generated images align with the text prompts by comparing their representations in CLIP's embedding space. A higher mean CLIP text alignment score indicates better text-to-image consistency.

Here's a breakdown of the key parts:
* CLIP Datasets (`CLIPCapDataset`, `CLIPImageDataset`): these classes are custom PyTorch Dataset implementations designed to prepare text captions and generated images for input into the CLIP model. They handle tokenization of captions and preprocessing of images (resizing, cropping, normalization) to match CLIP's expected input format.
* Feature Extraction (`extract_all_captions`, `extract_all_images`): these functions use the loaded CLIP model to extract numerical feature vectors (embeddings) from both the text captions and the generated images.
* Compute CLIPScore (`get_clip_score`): this function calculates the CLIPScore, which is a metric for text-image similarity. It first normalizes the extracted text and image features and then computer the cosine similarity between the normalized text and image feature vectors.
* Apply CLIPScore Formula: it scales the cosine similarity and potentially clips negative values.
* Calculate Mean Score: computes the average CLIPScore across all image-caption pairs to get an overall text alignment score.
* Main Eval (`clipeval_from_txt`): this is the main function that orchestrates the text alignment evaluation.





In [ ]:
import os
import json
import numpy as np
import torch
import clip
from pathlib import Path
from PIL import Image
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize

# --- CLIP Datasets ---
class CLIPCapDataset(torch.utils.data.Dataset):
    def __init__(self, data, prefix=""):
        self.data = data
        self.prefix = prefix + " " if prefix and not prefix.endswith(" ") else prefix

    def __getitem__(self, idx):
        cap = self.prefix + self.data[idx]
        cap = clip.tokenize(cap, truncate=True).squeeze()
        return {"caption": cap}

    def __len__(self):
        return len(self.data)

class CLIPImageDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
        self.preprocess = Compose([
            Resize(224, interpolation=Image.BICUBIC),
            CenterCrop(224),
            lambda x: x.convert("RGB"),
            ToTensor(),
            Normalize((0.48145466, 0.4578275, 0.40821073),
                      (0.26862954, 0.26130258, 0.27577711))
        ])

    def __getitem__(self, idx):
        img = Image.open(self.data[idx])
        return {"image": self.preprocess(img)}

    def __len__(self):
        return len(self.data)

# --- Feature extraction ---
def extract_all_captions(captions, model, device, batch_size=64):
    loader = torch.utils.data.DataLoader(
        CLIPCapDataset(captions),
        batch_size=batch_size, shuffle=False
    )
    feats = []
    with torch.no_grad():
        for b in loader:
            caps = b["caption"].to(device)
            feats.append(model.encode_text(caps).cpu().numpy())
    return np.vstack(feats)

def extract_all_images(images, model, device, batch_size=64):
    data = torch.utils.data.DataLoader(
        CLIPImageDataset(images),
        batch_size=batch_size, shuffle=False
    )
    all_image_features = []
    with torch.no_grad():
        for b in data:
            b = b['image'].to(device)
            if hasattr(model, 'encode_image'):
                if device == 'cuda':
                    b = b.to(torch.float16)
                all_image_features.append(model.encode_image(b).cpu().numpy())
            else:
                all_image_features.append(model(b).cpu().numpy())
    return np.vstack(all_image_features)



# --- Compute CLIPScore ---
def get_clip_score(model, image_feats, text_feats, w=2.5):
    # Normalize
    image_feats = image_feats / np.linalg.norm(image_feats, axis=1, keepdims=True)
    text_feats = text_feats / np.linalg.norm(text_feats, axis=1, keepdims=True)

    # Cosine similarity
    sim = np.sum(image_feats * text_feats, axis=1)

    # Apply CLIPScore formula
    per_sample = w * np.clip(sim, 0, None)   # clip negatives to 0
    return np.mean(per_sample), per_sample

# --- Main eval ---
def clipeval_from_txt(image_dir, prompts_txt, replacement, device="cuda", placeholder="{}"):
    # Collect image paths (sorted so they align with caption order)
    image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir)
                          if f.endswith((".png", ".jpg", ".jpeg"))])
    image_ids = [Path(p).stem for p in image_paths]

    # Load prompts from text file and replace placeholder
    with open(prompts_txt, "r") as f:
        captions = [line.strip().replace(placeholder, replacement) for line in f.readlines()]

    assert len(captions)*5 == len(image_paths), \
        f"Mismatch: {len(captions)} captions but {len(image_paths)} images."

    # Load CLIP
    model, _ = clip.load("ViT-B/32", device=device, jit=False)
    model.eval()

    repeats = len(image_paths) // len(captions)

    # Features
    image_feats = extract_all_images(image_paths, model, device)
    text_feats = extract_all_captions(captions, model, device)

    text_feats = np.repeat(text_feats, repeats, axis=0)

    # Score
    mean_score, per_sample = get_clip_score(model, image_feats, text_feats)
    print(f"Mean CLIP text alignment score: {mean_score:.4f}")
    return mean_score, per_sample, image_ids, captions

In [ ]:
mean_score, per_sample, ids, caps = clipeval_from_txt(gen_dir, prompt_file, replacement=f"<new1> {class_name}")

**Image alignment**

As Text alignment, Image alignment is based on the use of the cosine similarity computed between the embeddings of the generated images and the reference images, obtained with the CLIP model, to measure how well images resemble the target object.

In [ ]:
def clipeval_image(gen_image_dir, ref_image_dir, device="cuda"):
    # 1. Collect generated and reference images
    gen_paths = [os.path.join(gen_image_dir, f) for f in os.listdir(gen_image_dir)
                 if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    ref_paths = [os.path.join(ref_image_dir, f) for f in os.listdir(ref_image_dir)
                 if f.lower().endswith((".png", ".jpg", ".jpeg"))]

    # 2. Load CLIP
    model, _ = clip.load("ViT-B/32", device="cuda", jit=False)
    model.eval()

    # 3. Extract features
    gen_feats = extract_all_images(gen_paths, model, device)
    ref_feats = extract_all_images(ref_paths, model, device)

    # 4. Normalize
    gen_feats = gen_feats / np.linalg.norm(gen_feats, axis=1, keepdims=True)
    ref_feats = ref_feats / np.linalg.norm(ref_feats, axis=1, keepdims=True)

    # 5. Cosine similarity between generated and reference (matrix product)
    sims = gen_feats @ ref_feats.T  # [#gen, #ref]

    # 6. Average over all pairs
    return np.mean(sims)

In [ ]:
mean_score = clipeval_image(gen_dir, instance_directory, device="cuda")
print(f"Mean CLIP image alignment score: {mean_score:.4f}")

**Testing overfitting**

The following snippet of code is used to generate images of the same class of the instance to verify that the model is still able to "remember" the general features of the class and thus cope with language drift.

We decided to evaluate them through human evaluation instead of KID since Custom Diffusion uses a validation set of 500 real images coming from LAION-400M. However, since we use COCO we were limited in the number of available images for our chosen class and thus we would have re-used the same images of the training dataset. This of course would make the measure not entirely reliable.

In [ ]:
from itertools import cycle
import os

evaluation_prompts = "/content/drive/MyDrive/custom_diffusion/evaluation/prompts/motorbike.txt"
output_dir =  "/content/drive/MyDrive/custom_diffusion/evaluation/custom/motorbike1_exp3"

# Open the file and read lines into a list
with open(evaluation_prompts, 'r') as f:
    evaluation_prompts = f.readlines()

# Optional: Remove trailing newline characters from each line
prompts = [prompt.strip() for prompt in evaluation_prompts]

# Ensure the generation directory exists
os.makedirs(output_dir, exist_ok=True)

i = 0
for prompt in cycle(prompts):
  if i == 10:
    break

  # Generate an image
  # You can adjust parameters like num_inference_steps and guidance_scale
  image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]

  # Display the generated image
  image.save(os.path.join(output_dir, f"{class_name}_{i}.png"))
  #display(image)
  i = i + 1

## Results and analysis: tables, metrics and interpretations

###Hardware

To perform both the fine-tuning process and the image generation we exploited the **T4 GPU** available on Colab.

###Experiments run and Results

Due to the limitations explained in the Limitations section, we focused our evaluation on three subjects: a flower, a cup and a motorbike.
<br>
<br>

**Instance images**:

<img src="https://drive.google.com/uc?export=view&id=1BSYW19EcIlHqr8bz5VoSdFkTBMrsCFpq" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1pjqa1wvu2Fb933zVddXN8gcrVZxZSWLU" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=15OBB35ah1cDPLdOIdLjI2o-k60ftIzL_" width="300" height="300">

For both the Custom Diffusion approach and the DreamBooth approach we started by training our model with 250 steps on a training dataset composed by the same proportion of target images (obtained through data augmentation) and regularization images. As we can see the generated images are not too bad but the when generating images of the generic class, the model doesn't generalize well.
<br>
<br>

**Custom Diffusion**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.6021} & \text{0.686} & \color{red}{\text{Bad}}\\
\hline
\text{Cup} & \text{0.5972} & \text{0.7207} & \text{Mid} \\
\hline
\text{Motorbike} & \text{0.6055} & \text{0.7109} & \text{Mid}\\
\hline
\end{array}
$$
<br>

*Target images generated*

<table>
  <tr>
    <td align="center">
      <!-- flower_80-->
      <img src="https://drive.google.com/uc?export=view&id=1EL-xm-18o9_HjNYuJzhHXMy2ksU1vItb" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_19-->
      <img src="https://drive.google.com/uc?export=view&id=1qi55_h1C4jLy0YBOkLJR1ST6HSvgorIU" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_51 -->
      <img src="https://drive.google.com/uc?export=view&id=1vCQUG0vh_x7VxwDGmAW8NlTsPb0-95yw" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------


<table>
  <tr>
    <!-- cup_20 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=12ZoZyC8Tt-OyaWSIL1xsWUhy6CqYZ5Oz" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_58 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1_BVYcG07x1Ik0ls2poYby2q8PEh11Kd3" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_56 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1IlW4ufwQ-nf2YlfAOk24tTRctmpwp7Lt" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------


<table>
  <tr>
    <!-- motorbike_00-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1wus5vdMS12a-uspBHSlcXtamVMrDqU8_" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_17-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=136DsQXhSZ8kA3Ma43iNWRapZFJclxcSO" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_92 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1lGc4Bt13BR53mGIEkomGtnqy_we3ED4v" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=1_tI-lE6GZjJXG74owpprcuLZnKxWklY_" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1xZdNmtnMRFtHi_YiAwtJ5Utgvy0u3XIx" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1Zpign_Z0__BnVTBbJK6mhtsQKUFyf_f9" width="300" height="300">

**Prior preservation**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.5864} & \text{0.6875} & \color{red}{\text{Bad}}\\
\hline
\text{Cup} & \text{0.6069} & \text{0.7275} & \text{Mid} \\
\hline
\text{Motorbike} & \text{0.6499} & \text{0.7188} & \color{red}{\text{Bad}}\\
\hline
\end{array}
$$

<br>

*Target images generated*


<table>
  <tr>
    <td align="center">
      <!-- flower_00-->
      <img src="https://drive.google.com/uc?export=view&id=1kDcNCNP2VUKmMolvtqErnx7w4bGIMSgT" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_39-->
      <img src="https://drive.google.com/uc?export=view&id=1MsaBBJ8rF9bbftX8XPT1-F__AZ5y0Lyp" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_31 -->
      <img src="https://drive.google.com/uc?export=view&id=1RKX9X4WBTeBXxQjtKWIev9TiOblcQh8H" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------


<table>
  <tr>
    <!-- cup_40 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1Df_phVd0dl29x-ZtJJ3VxDyHFNb3zzUK" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_78 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=18lQvw1-Lbd9XmzIB1soLR6hgBDRoHefM" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_96 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1DHmFSWnJTqIk2jRQN5AHrVPCN3q4bFf2" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------


<table>
  <tr>
    <!-- motorbike_00-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1R4OXvNMXvP5gVOeNowfmnlMYxH2kLJTF" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_37-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1fLrUoVP-WLGo-PoOUafzvBnau8BGmdqT" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_52 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1QxEGJRyQeP8gnZeosqy8Q8IGw4FJ7LU0" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=14hLaFg2gpvV0Su6vDrlzwdkOuGVX5u93" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1iKwYqjLp0bPqumzl6LLeWOliIYO1xiQ6" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1WxjXvNlwV-7dPk1mCtXfMNoLXPDc9RPz" width="300" height="300">

<br>
<br>


Hoping to improve the generalization of our model, we tried modifying the ratio between target/regularization images, in such a way that the target images would result in just half of those of regularization. The results show an improvement in the generalization, at the cost of a worse quality of generated target images.

<br>

**Custom Diffusion**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.5864} & \text{0.6904} & \text{Mid}\\
\hline
\text{Cup} & \text{0.5991} & \text{0.7026} & \color{green}{\text{Good}} \\
\hline
\text{Motorbike} & \text{0.6147} & \text{0.7207} & \text{Mid}\\
\hline
\end{array}
$$

<br>

*Target images generated*


<table>
  <tr>
    <td align="center">
      <!-- flower_20-->
      <img src="https://drive.google.com/uc?export=view&id=1mqAtnozCfRaw1IZdHE_mfmxVDEJLRQC5" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_99-->
      <img src="https://drive.google.com/uc?export=view&id=1_98WCYC8qZlkAo_vCwNWSb0U0y-iVBra" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_11 -->
      <img src="https://drive.google.com/uc?export=view&id=1DqEZWjFz-xwhTYY9SMpxPJzOXsqwggIa" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------


<table>
  <tr>
    <!-- cup_60 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1UlqJyE4CaveBtIZIyo9875fANi4qAeWb" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_18 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1JaCqt54u6uk7mjy_HY6wfi84oWQ3-XQw" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_56 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1eS2U-PnnfEHG9lhV2HGoTueXCvYnWDIQ" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------


<table>
  <tr>
    <!-- motorbike_40-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1_iXI0siGLZf3Q2wIg16eFLltp6zKwN8U" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_57-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=15NW_gJpaBUDQ5_lZbfju13EyYgV9T4DB" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_32 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1N7i0XkBM0-XGEyBgRf_7Ypyn2eI1Z7j4" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=1zVMCa4q5F3q8K6DMu5FJrgoB4nBwnoNp" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1hepl_YivnkEt05_lo3WYL2q1QoQrnraD" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1h60aPBnW3iFWL065lpuG5oUagMPf_dQE" width="300" height="300">

<br>

**Prior preservation**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.5962} & \text{0.6729} & \color{green}{\text{Good}}\\
\hline
\text{Cup} & \text{0.5981} & \text{0.7065} & \color{green}{\text{Good}} \\
\hline
\text{Motorbike} & \text{0.605} & \text{0.6997} & \text{Mid}\\
\hline
\end{array}
$$

<br>

*Target images generated*

<table>
  <tr>
    <td align="center">
      <!-- flower_60-->
      <img src="https://drive.google.com/uc?export=view&id=1LbhGQeRmBE8spQ5SotYP3zyaUERMznmI" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_59-->
      <img src="https://drive.google.com/uc?export=view&id=1zkZXUKdSEPMHZUs5ElDyGZ5tFfzy0tSi" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_11 -->
      <img src="https://drive.google.com/uc?export=view&id=1tCpVDi_RBpON2ZfOqVsa0GE39kDUDkHH" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------

<table>
  <tr>
    <!-- cup_60 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1yIR3fWCDJ9yFbBO9Q8CFvCA0F7TF_3Tt" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_18 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1eJiH8IjS7mBqPINpCpYm_Tlw6sQtsLYY" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_56 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1sgzWDQzUokSZ27kz5UDGajwqo6mluGw3" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------

<table>
  <tr>
    <!-- motorbike_40-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1LsB7Sef4Vh7UgCLDK-lLbUSKikQvmsb3" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_57-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1cVmr8QDDJ72Jfjq6fr8dRFY5Foe20Gh-" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_32 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1qbjfW23KdrL2Tl0ZBSbgd3aSD_f_maVt" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=1NjxdcjqCXO1wv1F5PbM8wII8m3yBXIrZ" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1VHLdh8SRRUxiuy1jWCeL1EyI04DTVKXY" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1XzYMk0l-BFJAuWPYlReCpsT8QB5FrIVL" width="300" height="300">

Since we generally observed an improvement for what concerns overfitting in the second approach, we tried increasing the number of training steps to 500, hoping this would help the model both learning better the target item, while mantaining a good generalization. The outcome was as expected.

<br>

**Custom Diffusion**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.5972} & \text{0.7046} & \color{green}{\text{Good}}\\
\hline
\text{Cup} & \text{0.6069} & \text{0.7148} & \text{Mid} \\
\hline
\text{Motorbike} & \text{0.6245} & \text{0.7563} & \color{green}{\text{Good}}\\
\hline
\end{array}
$$

<br>

*Target images generated*

<table>
  <tr>
    <td align="center">
      <!-- flower_60-->
      <img src="https://drive.google.com/uc?export=view&id=1z1x-XMDXTyLWcfZrdaTYc5KVd3y4g75-" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_59-->
      <img src="https://drive.google.com/uc?export=view&id=1-cyslOii5JmTUI39fhzpRMudn_e5Lo_J" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_11 -->
      <img src="https://drive.google.com/uc?export=view&id=1yZ7EG0wd7EYTij42v-CPWHkTOKeEI3ZC" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------

<table>
  <tr>
    <!-- cup_60 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1tQxINzIsMlE8QvB-U5K5vlfVHUqdGs_u" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_18 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=113-S0fyaQZYeMW5EDWPAknnfWu9O-0JQ" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_56 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1wx2BUZNmlTuyXgrQesfwPdoEPil7jstZ" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------

<table>
  <tr>
    <!-- motorbike_40-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1MBH0p1HDKRRkimmHKmxBucHg-opRfneX" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_57-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=17cWyAU4deoJ_m52o7_QV6ykfYyZDg2C7" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_32 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=18NEjxg2XjxdWfANFr47bm_SNOgM9NqJm" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=1DQ3G7icsc7Sro4w1au55OeE12TUkTvIz" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=13pxszggIx_oZOpBrMXSLnXOPDFLRa3HF" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1yc3C01fZJb2ZauuKBxss6d2LoaVOwF4L" width="300" height="300">

<br>

**Prior preservation**

<br>

$$
\begin{array}{|c|c|c|c|}
\hline
\text{Class name} & \text{Text alignment} & \text{Image alignment} & \text{Overfitting}\\
\hline
\text{Flower} & \text{0.5918} & \text{0.7007} & \color{green}{\text{Good}}\\
\hline
\text{Cup} & \text{0.6196} & \text{0.7061} & \text{Mid} \\
\hline
\text{Motorbike} & \text{0.6235} & \text{0.7383} & \text{Mid}\\
\hline
\end{array}
$$

<br>

*Target images generated*

<table>
  <tr>
    <td align="center">
      <!-- flower_60-->
      <img src="https://drive.google.com/uc?export=view&id=1Jmxza_CdlHoopY0_KMsrdOgy3KxmABCe" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; flower.</em>
    </td>
    <td align="center">
      <!-- flower_59-->
      <img src="https://drive.google.com/uc?export=view&id=14P2b8qPQ5s9CvAJxcAPGpUYBy4e_UbZ6" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower made of crystal.</em>
    </td>
    <td align="center">
      <!-- flower_11 -->
      <img src="https://drive.google.com/uc?export=view&id=1NIgERabh5kmz-zkQ6Nm5vMdgCXuiged2" width="300" height="300"><br>
      <em>A &lt;new1&gt; flower painting by artist claude monet.</em>
    </td>
  </tr>
</table>

-------------------------------------------------------------------------

<table>
  <tr>
    <!-- cup_60 photo of a {}. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1DuvZWNZw8EBhwAPitzKZQGC44cI4v26y" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; cup.</em>
    </td>
    <!-- cup_18 {} made of glass. -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1CfItigrZk29tVcyrXzA-t3kSFkZsfwor" width="300" height="300"><br>
      <em>A &lt;new1&gt; cup made of glass.</em>
    </td>
    <!-- cup_56 Georgia O'Keeffe style {} painting.-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1IjtEfHBTRg_zUV4OqIB6dD9afEQHCleV" width="300" height="300"><br>
      <em>Georgia O'Keeffe style &lt;new1&gt; cup painting.</em>
    </td>
  </tr>
</table>


-------------------------------------------------------------------------

<table>
  <tr>
    <!-- motorbike_40-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1CIRFKXZ9yd65rcCebpVgY7c2D3MGyD1-" width="300" height="300"><br>
      <em>A photo of a &lt;new1&gt; motorbike.</em>
    </td>
    <!-- motorbike_57-->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1DivZxqGv_AFCtIIqkXukHWL3ZkLbwSzp" width="300" height="300"><br>
      <em>A &lt;new1&gt; motorbike made of LEGO bricks.</em>
    </td>
    <!-- motorbike_32 -->
    <td align="center">
      <img src="https://drive.google.com/uc?export=view&id=1DdlntUbdiNdvuq4FxwY0BDussmOzERCG" width="300" height="300"><br>
      <em>A watercolor painting of &lt;new1&gt; motorbike, cruising down a countryside road.</em>
    </td>
  </tr>
</table>

<br>

*Class images*

<img src="https://drive.google.com/uc?export=view&id=1TtZqlDskcBzsI_l2UZ_BqDjE2We9Or1N" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1Iag8ZXYJGa_tKP_mIiq4aDGkjp1U3Fgl" width="300" height="300">
<img src="https://drive.google.com/uc?export=view&id=1MUKwNqDpxOr5bOkYkFOFTWSdRUT0i5w_" width="300" height="300">



## Limitations and reflections

###Limitations

The results of our model with respect to those obtained by DreamBooth and Custom Diffusion are lower and pretty limited and we suppose this is mainly due to two issues:


*   the dataset COCO used when adopting Custom Diffusion approach
*   Colab resources limits

 The extremely smaller size of COCO with respect to LAION-400M had a cost: just some of the subjects we wanted to test had a sufficient number of images in it and generally those images where not satisfying. Furthermore, COCO primary application field is computer vision, so the pictures were not always ideal for our task. For example, when looking for a photo of an object, lots of results pictured a totally different scene, with that item appearing maybe just in the background.
<br>
This probably did not help our model in its regularization.

The main obstacle however remains the limited environment offered by Colab, since it was difficult to generate a satisfying number of images with just the daily amount of GPU offered.
<br>
These forced us to:

*   limit the number of regularization images generated for DreamBooth
*   reduce the batch size to avoid to run out of memory on the GPU
*   test just the training of $W_{k}$ and $W_{v}$ parameters to avoid to run out of memory on the GPU
*   perform the evaluation just on 100 images per model (instead of the 1K used in the papers)


###Conclusions
In the following tables, we use these abbreviations:
* Experiment 1: 1/1 ratio, 250 steps;
* Experiment 2: 1/2 ratio, 250 steps;
* Experiment 3: 1/2 ratio, 500 steps.

<br>

**Text alignment**
<br>
$$
\begin{array}{|c|c|c|c|}
\hline
\text{Approach} & \text{Experiment 1} & \text{Experiment 2} & \text{Experiment 3}\\
\hline
\text{Custom} & \text{0.6016} & \text{0.6001} & \textbf{0.6095}\\
\hline
\text{Prior} & \textbf{0.6144} & \text{0.5998} & \text{0.6116} \\
\hline
\end{array}
$$

**Image alignment**
<br>
$$
\begin{array}{|c|c|c|c|}
\hline
\text{Approach} & \text{Experiment 1} & \text{Experiment 2} & \text{Experiment 3}\\
\hline
\text{Custom} & \text{0.7059} & \text{0.7046} & \textbf{0.7252}\\
\hline
\text{Prior} & \text{0.7113} & \text{0.693} & \textbf{0.715} \\
\hline
\end{array}
$$

<br>

In conclusion, in our project we tested two different approaches to personalize Text-to-Image diffusion models, also trying mixing them, such as using data augmentation in the DreamBooth mode.
<br>
As we can see, there is not a clear "winner" between the two approaches but we obtained in general good performances with respect to our task: generate good target images while mantaining a certain degree of generalization capabilities.
<br>
However, we observed that while the models improved their generalization in experiment 3, the quality of the generated images got a bit worse than  those from experiment 1.

###Suggestions for future work
Future possible ideas to improve our project would be:


*   look for a better dataset from which retrieve regularization images for Custom approach
*   use a more powerful hardware, in order to be able to generate more images both for the regularization dataset in DreamBooth approach, for the evaluation phase and to handle a bigger batch size during training
*   test the approaches on a wider range of subjects


## References (papers, code repositories, datasets)

Papers:


*   [DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation](https://openaccess.thecvf.com/content/CVPR2023/html/Ruiz_DreamBooth_Fine_Tuning_Text-to-Image_Diffusion_Models_for_Subject-Driven_Generation_CVPR_2023_paper.html#:~:text=given%20reference%20set%20and%20synthesize,our%20technique%20enables%20synthesizing%20the)
*   [Multi-Concept Customization of Text-to-Image Diffusion](https://openaccess.thecvf.com/content/CVPR2023/papers/Kumari_Multi-Concept_Customization_of_Text-to-Image_Diffusion_CVPR_2023_paper.pdf)

Code Repositories:


*   https://github.com/adobe-research/custom-diffusion
*   https://dreambooth.github.io
*   https://github.com/CompVis/stable-diffusion

Datasets:


*   https://cocodataset.org
*   https://www.cs.cmu.edu/~custom-diffusion/dataset.html


Others:


*   slides the Neural Networks course










## Reproducibility instructions (dependencies, Colab setup, etc.)

In this section are reported all the preliminary steps done in order to download and install everything that is needed to execute correctly the notebook. There are two possible ways of setting everything up.
The mount of the google drive needs to be done in both cases.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


### 1. Option 1 (preferred)
In case the user has access to the [`custom_diffusion`](https://drive.google.com/drive/folders/1ttK4_nl8wrU7zHSieYifSwIqBKpSgOjg?usp=sharing) folder, he/she can add it to its Google Drive and just run the following cell before executing the implementation code.

In [ ]:
%cd /content/drive/MyDrive/custom_diffusion/stable-diffusion
!pip install torch torchvision
!pip install transformers==4.49 diffusers==0.32.2 invisible-watermark tokenizers
!pip install -e .
!pip install pytorch-lightning==1.9.1
!pip install kornia
%cd ..
%cd /content/drive/MyDrive/custom_diffusion/taming-transformers
!pip install -e .
%cd ..
%cd clip
!pip install -e .
%cd ..
%cd stable-diffusion
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

### Option 2 (installation from scratch)
If a user wants to build its own custom_diffusion folder from scratch, he/she has to follows the steps reported below.

1. create a `custom_diffusion` folder.

In [ ]:
!mkdir /content/drive/MyDrive/custom_diffusion

2. download the dataset containing possible instances ready to be used to fine tune the model.

In [ ]:
# Instance images folder download
!pip install gdown
!gdown 1jj8JMtIS5-8vRtNtZ2x8isieWH9yetuK -O /content/drive/MyDrive/custom_diffusion/benchmark_dataset.zip
!unzip /content/drive/MyDrive/custom_diffusion/benchmark_dataset.zip -d /content/drive/MyDrive/custom_diffusion/

In [ ]:
!mkdir /content/drive/MyDrive/custom_diffusion/customconcept101/
# Dataset for single-concept case
!wget -O /content/drive/MyDrive/custom_diffusion/customconcept101/dataset.json https://raw.githubusercontent.com/adobe-research/custom-diffusion/main/customconcept101/dataset.json
# Dataset for multiconcept case
!wget -O /content/drive/MyDrive/custom_diffusion/customconcept101/dataset_multiconcept.json https://raw.githubusercontent.com/adobe-research/custom-diffusion/main/customconcept101/dataset_multiconcept.json

In [ ]:
# Download on the prompts to test the model's capabilities
# move in the content folder
%cd /content
# initialize a temporary repository
!git init temp-diffusion
# move in the temporary folder
%cd temp-diffusion
# add the remote
!git remote add origin https://github.com/adobe-research/custom-diffusion.git
# enable sparse checkout
!git config core.sparseCheckout true
# specify the folder you want to download
!echo "customconcept101/prompts/*" >> .git/info/sparse-checkout
# download only the specified folder
!git pull origin main
# move the folder prompts directly in /content/drive/MyDrive/custom_diffusion/customconcept101/prompts
!mv customconcept101/prompts /content/drive/MyDrive/custom_diffusion/customconcept101/prompts
# move in the main folder and remove the temporary folder
%cd /content/drive/MyDrive/custom_diffusion/customconcept101
!rm -rf /content/temp-diffusion

3. Download Stable Diffusion model

In [ ]:
#creare cartella custom_diffusion
%cd /content/drive/MyDrive/custom_diffusion
!git clone https://github.com/CompVis/stable-diffusion.git
!pip install torch torchvision
!pip install transformers==4.49 diffusers==0.32.2 invisible-watermark tokenizers
!pip install -e /content/drive/MyDrive/custom_diffusion/stable-diffusion
!pip install pytorch-lightning==1.9.1

!git clone https://github.com/CompVis/taming-transformers.git /content/drive/MyDrive/custom_diffusion/taming-transformers
!pip install -e /content/drive/MyDrive/custom_diffusion/taming-transformers

!git clone https://github.com/openai/CLIP.git /content/drive/MyDrive/custom_diffusion/clip
!pip install -e /content/drive/MyDrive/custom_diffusion/clip

!pip install kornia

!wget https://huggingface.co/CompVis/stable-diffusion-v-1-4-original/resolve/main/sd-v1-4.ckpt
%cd /content/drive/MyDrive/custom_diffusion/stable-diffusion/
!rm -rf models/ldm/stable-diffusion-v1/
!mkdir -p models/ldm/stable-diffusion-v1/
!mv /content/drive/MyDrive/custom_diffusion/stable-diffusion/sd-v1-4.ckpt models/ldm/stable-diffusion-v1/model.ckpt

 Before testing that everything has been downloaded correctly, change file "/content/drive/MyDrive/custom_diffusion2/stable-diffusion/scripts/txt2img.py", line 51, in load_model_from_config from
 `pl_sd = torch.load(ckpt, map_location="cpu") ` with <br>
 `pl_sd = torch.load(ckpt, map_location="cpu", weights_only=False)`

In [ ]:
#to test that everything is installed correctly
!python scripts/txt2img.py --prompt "a photograph of an astronaut riding a horse" --plms

4. Download the COCO dataset to later retrieve the regularization images

In [ ]:
!mkdir /content/drive/MyDrive/custom_diffusion2/coco
!wget http://images.cocodataset.org/zips/train2017.zip
!unzip train2017.zip -d /content/drive/MyDrive/custom_diffusion2/coco/train2017
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip annotations_trainval2017.zip -d /content/drive/MyDrive/custom_diffusion2/coco/annotations

These are all one-time operations. Once the user has completed them, he/she can refer to Option 1 instructions.